# Beispiel: Residual Block mit 6 ConvsBeispiel: Residual Block mit 6 Convs
Wichtige Punkte

**Add-Operation findet nur einmal pro Block statt**.

Innerhalb des Blocks können beliebig viele Convs sein (2–6 in deinem Beispiel).

Standard ResNet-Architektur:

BasicBlock: 2 Convs

BottleneckBlock: 3 Convs (1x1 → 3x3 → 1x1)

## ReLU/BatchNorm kann nach jedem Conv liegen – das ändert nichts am Prinzip.

ResNet-34/18, oft BasicBlock genannt.

BasicBlock: 2 Conv-Layer, danach Add + ReLU

BottleneckBlock (ResNet-50/101/152): 3 Convs (1x1 → 3x3 → 1x1)


In [2]:
import torch
import torch.nn as nn

class ResidualBlock6(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        identity = x
        out = self.convs(x)   # F(x_l) = 6 Convs + ReLUs
        out = out + identity  # x_l + F(x_l)
        out = self.relu(out)
        return out

# Test
x = torch.randn(1, 64, 56, 56)
block6 = ResidualBlock6(64)
y = block6(x)
print("Input shape :", x.shape)
print("Output shape:", y.shape)

Input shape : torch.Size([1, 64, 56, 56])
Output shape: torch.Size([1, 64, 56, 56])


# Mini-ResNet in PyTorch
ein Mini-ResNet in PyTorch, das die ResNet-Architektur-Prinzipien zeigt: mehrere Residual Blocks, Shortcut Additionen, Pooling und Klassifizierung. Ich halte es bewusst klein, damit es überschaubar bleibt, ähnlich wie ResNet-18, aber mit minimalen Channels für Demo.


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------- Residual Block (BasicBlock, 2 Convs) --------
class BasicBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += identity     # Shortcut Addition
        out = self.relu(out)
        return out

# -------- Mini-ResNet --------
class MiniResNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2)  # 128x128 -> 64x64
        )

        # 4 Residual Blocks
        self.layer1 = BasicBlock(16)
        self.layer2 = BasicBlock(16)
        self.layer3 = BasicBlock(16)
        self.layer4 = BasicBlock(16)

        # Klassifizierung
        self.global_pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(16, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.global_pool(x)      # -> [B,16,1,1]
        x = torch.flatten(x, 1)      # -> [B,16]
        x = self.fc(x)               # -> [B,num_classes]
        return x

## Testlauf

In [4]:
x = torch.randn(2, 3, 128, 128)  # Batch=2, RGB, 128x128
model = MiniResNet(num_classes=10)
y = model(x)
print("Output shape:", y.shape)

Output shape: torch.Size([2, 10])
